# Modelos de Regresión Lineal basado en productos clusterizados por DTW

In [2]:
# EJECUTAR LIBRERIAS
import os
import pandas as pd

In [3]:
# LEEMOS LOS DATOS
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Data/'
filename = 'sell-in.txt'
filepath = os.path.join(drive_base_path, filename)
sell = pd.read_csv(filepath, sep='\t')
print(sell.head(10))

filename = 'product_id_apredecir201912.txt'
filepath = os.path.join(drive_base_path, filename)
a_predecir = pd.read_csv(filepath, sep='\t')

   periodo  customer_id  product_id  plan_precios_cuidados  cust_request_qty  \
0   201701        10234       20524                      0                 2   
1   201701        10032       20524                      0                 1   
2   201701        10217       20524                      0                 1   
3   201701        10125       20524                      0                 1   
4   201701        10012       20524                      0                11   
5   201701        10080       20524                      0                 1   
6   201701        10015       20524                      0                 4   
7   201701        10062       20524                      0                 1   
8   201701        10159       20524                      0                 3   
9   201701        10183       20524                      0                 1   

   cust_request_tn       tn  
0          0.05300  0.05300  
1          0.13628  0.13628  
2          0.03028  0.03028  

In [4]:
# Veo la cantidad de valores distintos en "periodo"
num_periodos = sell['periodo'].nunique()
print(f'Cantidad de valores distintos en "periodo": {num_periodos}')

Cantidad de valores distintos en "periodo": 36


In [5]:
# Agrupar y sumar
sell_agrup = (
    sell
    .groupby(['periodo', 'product_id'], as_index=False)['tn']
    .sum()
)

print(sell_agrup)

       periodo  product_id          tn
0       201701       20001   934.77222
1       201701       20002   550.15707
2       201701       20003  1063.45835
3       201701       20004   555.91614
4       201701       20005   494.27011
...        ...         ...         ...
31238   201912       21265     0.05007
31239   201912       21266     0.05121
31240   201912       21267     0.01569
31241   201912       21271     0.00298
31242   201912       21276     0.00892

[31243 rows x 3 columns]


In [6]:
# FEATURE ENGINEERING Y CREACION DE LA CLASE A PREDECIR
# Ordenar por product_id y periodo para asegurar el orden correcto
sell_agrup = sell_agrup.sort_values(['product_id', 'periodo']).reset_index(drop=True)

# Crear las 11 columnas con los valores de los períodos anteriores
for i in range(1, 12):  # Del 1 al 11
    sell_agrup[f'tn_lag_{i}'] = sell_agrup.groupby('product_id')['tn'].shift(i)

# Agregar columna con el valor de tn del período +2 (2 períodos hacia adelante)
sell_agrup['tn_target'] = sell_agrup.groupby('product_id')['tn'].shift(-2)

# Mostrar el resultado
print("Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:")
print(sell_agrup.head(36))
print(f"\nForma del dataset: {sell_agrup.shape}")
print(f"Columnas: {list(sell_agrup.columns)}")

Dataset con las 11 columnas de períodos anteriores y 1 columna de período futuro:
    periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
0    201701       20001   934.77222         NaN         NaN         NaN   
1    201702       20001   798.01620   934.77222         NaN         NaN   
2    201703       20001  1303.35771   798.01620   934.77222         NaN   
3    201704       20001  1069.96130  1303.35771   798.01620   934.77222   
4    201705       20001  1502.20132  1069.96130  1303.35771   798.01620   
5    201706       20001  1520.06539  1502.20132  1069.96130  1303.35771   
6    201707       20001  1030.67391  1520.06539  1502.20132  1069.96130   
7    201708       20001  1267.39462  1030.67391  1520.06539  1502.20132   
8    201709       20001  1316.94604  1267.39462  1030.67391  1520.06539   
9    201710       20001  1439.75563  1316.94604  1267.39462  1030.67391   
10   201711       20001  1580.47401  1439.75563  1316.94604  1267.39462   
11   201712       

In [7]:
# Definir la lista de product_id específicos
magicos = [20002, 20001, 20003, 20004, 20005, 20006, 20010, 20014, 20007, 20019, 20013, 20011, 20015, 20008, 20026, 
           20016, 20023, 20020, 20012, 20017, 20021, 20018, 20024, 20025, 20031, 20042, 20046, 20049, 20028, 20029, 
           20035, 20045, 20044, 20033, 20054, 20053, 20038, 20039, 20059, 20047, 20041, 20061, 20051, 20043, 20075, 
           20050, 20063, 20057, 20058, 20073, 20052, 20065, 20069, 20070, 20056, 20094, 20081, 20062, 20068, 20071, 
           20066, 20055, 20076, 20037, 20112, 20067, 20093, 20030, 20080, 20091, 20107, 20074, 20101, 20095, 20087, 
           20106, 20145, 20119, 20108, 20077, 20122, 20103, 20158, 20153, 20157, 20123, 20134, 20125, 20118, 20132, 
           20155, 20164, 20140, 20099, 20082, 20092, 20072, 20129, 20139, 20148, 20114, 20133, 20096, 20161, 20162, 
           20086, 20144, 20097, 20203, 20100, 20124, 20151, 20146, 20109, 20137, 20167, 20117, 20090, 20235, 20079, 
           20189, 20166, 20138, 20181, 20142, 20177, 20102, 20212, 20297, 20175, 20176, 20184, 20306, 20193, 20196, 
           20179, 20218, 20316, 20188, 20182, 20201, 20187, 20198, 20321, 20205, 20197, 20200, 20152, 20323, 20202, 
           20220, 20168, 20233, 20208, 20313, 20215, 20206, 20209, 20238, 20219, 20231, 20222, 20251, 20267, 20290, 
           20225, 20211, 20340, 20263, 20244, 20276, 20254, 20256, 20303, 20230, 20246, 20337, 20361, 20226, 20268, 
           20264, 20239, 20241, 20289, 20311, 20240, 20253, 20275, 20282, 20307, 20288, 20352, 20314, 20300, 20296, 
           20207, 20299, 20334, 20284, 20343, 20324, 20417, 20377, 20281, 20450, 20384, 20388, 20473, 20483, 20520, 
           20565, 20652, 20862, 20878, 20843, 20852, 21192]

# Crear el subconjunto filtrado por período 201812 y los product_id específicos
sell_agrup_subset = sell_agrup[
    (sell_agrup['periodo'] == 201812) & 
    (sell_agrup['product_id'].isin(magicos))
]

print(f"Dataset original shape: {sell_agrup.shape}")
print(f"Subconjunto filtrado shape: {sell_agrup_subset.shape}")
print(f"Cantidad de productos únicos en el subconjunto: {sell_agrup_subset['product_id'].nunique()}")
print("\nPrimeras filas del subconjunto:")
print(sell_agrup_subset.head())

Dataset original shape: (31243, 15)
Subconjunto filtrado shape: (217, 15)
Cantidad de productos únicos en el subconjunto: 217

Primeras filas del subconjunto:
     periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
23    201812       20001  1486.68669  1813.01511  2295.19832  1438.67455   
59    201812       20002  1009.45458  1766.81068  1378.49032   954.23575   
95    201812       20003   769.82869  1206.91773  1313.34211   912.34156   
131   201812       20004   585.56477   802.34669   809.67086   948.86342   
167   201812       20005   372.63428   469.26344   893.74086   761.77520   

       tn_lag_4    tn_lag_5    tn_lag_6    tn_lag_7    tn_lag_8    tn_lag_9  \
23   1800.96168  1470.41009  1150.79169  1293.89788  1251.28462  1856.83534   
59   1161.88430   977.40239  1033.82845  1103.39191   999.20934   966.86044   
95    955.97079   656.22700   660.73323   784.35885   765.47838   778.55594   
131   936.42001   653.42310   447.84475   641.37063   611.51237   48

In [17]:
# Realizar merge entre sell_agrup_subset y clusters usando 'product_id'
clusters = pd.read_csv(os.path.join(
    'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Dinamic Time Warping (DTW)', 'clusters_dtw_2018.csv'), 
    sep = '/')
sell_agrup_subset_clusters = sell_agrup_subset.merge(clusters, on='product_id', how='left')

# Mostrar las primeras filas del resultado
print(sell_agrup_subset_clusters.head())
print(f"Shape del dataframe resultante: {sell_agrup_subset_clusters.shape}")
print(f"Columnas: {list(sell_agrup_subset_clusters.columns)}")

   periodo  product_id          tn    tn_lag_1    tn_lag_2    tn_lag_3  \
0   201812       20001  1486.68669  1813.01511  2295.19832  1438.67455   
1   201812       20002  1009.45458  1766.81068  1378.49032   954.23575   
2   201812       20003   769.82869  1206.91773  1313.34211   912.34156   
3   201812       20004   585.56477   802.34669   809.67086   948.86342   
4   201812       20005   372.63428   469.26344   893.74086   761.77520   

     tn_lag_4    tn_lag_5    tn_lag_6    tn_lag_7    tn_lag_8    tn_lag_9  \
0  1800.96168  1470.41009  1150.79169  1293.89788  1251.28462  1856.83534   
1  1161.88430   977.40239  1033.82845  1103.39191   999.20934   966.86044   
2   955.97079   656.22700   660.73323   784.35885   765.47838   778.55594   
3   936.42001   653.42310   447.84475   641.37063   611.51237   488.92473   
4   874.88924   502.34077   547.62513   637.11135   496.41774   559.98671   

    tn_lag_10   tn_lag_11   tn_target  cluster  
0  1043.76470  1169.07532  1259.09363      

In [18]:
# Importar librerías necesarias para regresión lineal
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

print("=== ENTRENAMIENTO DE MODELOS POR CLUSTER ===")

# Usar el dataset con clusters
data_with_clusters = sell_agrup_subset_clusters.copy()

# Preparar las columnas (mismas que antes pero del dataset con clusters)
feature_columns = ['tn', 'tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_4', 'tn_lag_5', 
                   'tn_lag_6', 'tn_lag_7', 'tn_lag_8', 'tn_lag_9', 'tn_lag_10', 'tn_lag_11']
target_column = 'tn_target'

print(f"Variables predictoras: {feature_columns}")
print(f"Variable target: {target_column}")
print(f"Dataset shape: {data_with_clusters.shape}")

# Verificar clusters disponibles
clusters_disponibles = data_with_clusters['cluster'].unique()
print(f"Clusters disponibles: {sorted(clusters_disponibles)}")

# Diccionario para almacenar modelos y métricas por cluster
modelos_por_cluster = {}
metricas_por_cluster = {}

# Entrenar un modelo para cada cluster
for cluster_id in sorted(clusters_disponibles):
    print(f"\n=== CLUSTER {cluster_id} ===")
    
    # Filtrar datos del cluster
    datos_cluster = data_with_clusters[data_with_clusters['cluster'] == cluster_id].copy()
    print(f"Productos en cluster {cluster_id}: {len(datos_cluster)}")
    
    # Preparar X e y para este cluster
    X_cluster = datos_cluster[feature_columns]
    y_cluster = datos_cluster[target_column]
    
    # Eliminar filas con valores NaN
    mask_cluster = ~(X_cluster.isnull().any(axis=1) | y_cluster.isnull())
    X_cluster_clean = X_cluster[mask_cluster]
    y_cluster_clean = y_cluster[mask_cluster]
    
    print(f"Datos sin NaN - X: {X_cluster_clean.shape}, y: {y_cluster_clean.shape}")
    
    # Verificar si hay suficientes datos para entrenar
    if len(X_cluster_clean) >= 2:  # Mínimo 2 muestras para entrenar
        try:
            # Entrenar modelo de regresión lineal para este cluster
            modelo_cluster = LinearRegression()
            modelo_cluster.fit(X_cluster_clean, y_cluster_clean)
            
            # Hacer predicciones
            y_pred_cluster = modelo_cluster.predict(X_cluster_clean)
            
            # Calcular métricas
            r2_cluster = r2_score(y_cluster_clean, y_pred_cluster) if len(y_cluster_clean) > 1 else 0
            mse_cluster = mean_squared_error(y_cluster_clean, y_pred_cluster)
            mae_cluster = mean_absolute_error(y_cluster_clean, y_pred_cluster)
            rmse_cluster = np.sqrt(mse_cluster)
            
            # Almacenar modelo y métricas
            modelos_por_cluster[cluster_id] = modelo_cluster
            metricas_por_cluster[cluster_id] = {
                'r2': r2_cluster,
                'mse': mse_cluster,
                'rmse': rmse_cluster,
                'mae': mae_cluster,
                'n_samples': len(X_cluster_clean),
                'productos': list(datos_cluster['product_id'].values)
            }
            
            print(f"✅ Modelo entrenado exitosamente")
            print(f"   R² Score: {r2_cluster:.4f}")
            print(f"   MSE: {mse_cluster:.4f}")
            print(f"   RMSE: {rmse_cluster:.4f}")
            print(f"   MAE: {mae_cluster:.4f}")
            print(f"   Muestras de entrenamiento: {len(X_cluster_clean)}")
            
            # Mostrar algunos coeficientes más importantes
            coeficientes = modelo_cluster.coef_
            intercepto = modelo_cluster.intercept_
            print(f"   Intercepto: {intercepto:.4f}")
            
            # Mostrar los 3 coeficientes más importantes (en valor absoluto)
            coef_abs = [(abs(coef), i, coef) for i, coef in enumerate(coeficientes)]
            coef_abs.sort(reverse=True)
            print(f"   Top 3 coeficientes:")
            for abs_val, idx, coef_val in coef_abs[:3]:
                print(f"     {feature_columns[idx]}: {coef_val:.4f}")
                
        except Exception as e:
            print(f"❌ Error al entrenar modelo para cluster {cluster_id}: {e}")
            
    else:
        print(f"⚠️  Insuficientes datos para entrenar (mínimo 2 muestras requeridas)")

print(f"\n=== RESUMEN DE MODELOS ENTRENADOS ===")
print(f"Total clusters procesados: {len(clusters_disponibles)}")
print(f"Modelos entrenados exitosamente: {len(modelos_por_cluster)}")

if metricas_por_cluster:
    print(f"\n=== COMPARACIÓN DE RENDIMIENTO POR CLUSTER ===")
    print(f"{'Cluster':<8} {'Productos':<10} {'Muestras':<9} {'R²':<8} {'RMSE':<8} {'MAE':<8}")
    print("-" * 60)
    
    for cluster_id in sorted(metricas_por_cluster.keys()):
        metrics = metricas_por_cluster[cluster_id]
        print(f"{cluster_id:<8} {len(metrics['productos']):<10} {metrics['n_samples']:<9} "
              f"{metrics['r2']:<8.4f} {metrics['rmse']:<8.2f} {metrics['mae']:<8.2f}")
    
    # Encontrar el mejor cluster por R²
    mejor_cluster = max(metricas_por_cluster.keys(), key=lambda x: metricas_por_cluster[x]['r2'])
    mejor_r2 = metricas_por_cluster[mejor_cluster]['r2']
    print(f"\n🏆 MEJOR CLUSTER: {mejor_cluster} (R² = {mejor_r2:.4f})")
    
    # Estadísticas generales
    r2_promedio = np.mean([m['r2'] for m in metricas_por_cluster.values()])
    rmse_promedio = np.mean([m['rmse'] for m in metricas_por_cluster.values()])
    print(f"📊 R² promedio entre clusters: {r2_promedio:.4f}")
    print(f"📊 RMSE promedio entre clusters: {rmse_promedio:.2f}")
else:
    print("❌ No se pudo entrenar ningún modelo por cluster")

=== ENTRENAMIENTO DE MODELOS POR CLUSTER ===
Variables predictoras: ['tn', 'tn_lag_1', 'tn_lag_2', 'tn_lag_3', 'tn_lag_4', 'tn_lag_5', 'tn_lag_6', 'tn_lag_7', 'tn_lag_8', 'tn_lag_9', 'tn_lag_10', 'tn_lag_11']
Variable target: tn_target
Dataset shape: (217, 16)
Clusters disponibles: [np.int64(0), np.int64(3), np.int64(7)]

=== CLUSTER 0 ===
Productos en cluster 0: 150
Datos sin NaN - X: (150, 12), y: (150,)
✅ Modelo entrenado exitosamente
   R² Score: 0.9456
   MSE: 506.9714
   RMSE: 22.5160
   MAE: 15.0192
   Muestras de entrenamiento: 150
   Intercepto: 7.8409
   Top 3 coeficientes:
     tn: 0.4734
     tn_lag_2: 0.2620
     tn_lag_10: 0.2486

=== CLUSTER 3 ===
Productos en cluster 3: 13
Datos sin NaN - X: (13, 12), y: (13,)
✅ Modelo entrenado exitosamente
   R² Score: 1.0000
   MSE: 0.0000
   RMSE: 0.0000
   MAE: 0.0000
   Muestras de entrenamiento: 13
   Intercepto: -1.1707
   Top 3 coeficientes:
     tn: -2.1565
     tn_lag_8: -1.5219
     tn_lag_7: 1.3978

=== CLUSTER 7 ===
Produc

In [19]:
# Filtrar sell_agrup por los product_id presentes en a_predecir
sell_agrup_filtrado = sell_agrup[sell_agrup['product_id'].isin(a_predecir['product_id'])]
print(sell_agrup_filtrado)

drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'sell_agrup_filtrado.txt'
filepath = os.path.join(drive_base_path, filename)

# Guardar el DataFrame filtrado en un archivo de texto
sell_agrup_filtrado.to_csv(filepath, sep='\t', index=False)

       periodo  product_id          tn    tn_lag_1    tn_lag_2   tn_lag_3  \
0       201701       20001   934.77222         NaN         NaN        NaN   
1       201702       20001   798.01620   934.77222         NaN        NaN   
2       201703       20001  1303.35771   798.01620   934.77222        NaN   
3       201704       20001  1069.96130  1303.35771   798.01620  934.77222   
4       201705       20001  1502.20132  1069.96130  1303.35771  798.01620   
...        ...         ...         ...         ...         ...        ...   
31206   201908       21276     0.01265     0.00223     0.04086    0.09283   
31207   201909       21276     0.01856     0.01265     0.00223    0.04086   
31208   201910       21276     0.02079     0.01856     0.01265    0.00223   
31209   201911       21276     0.03341     0.02079     0.01856    0.01265   
31210   201912       21276     0.00892     0.03341     0.02079    0.01856   

        tn_lag_4  tn_lag_5  tn_lag_6  tn_lag_7  tn_lag_8  tn_lag_9  tn_lag_

In [20]:
# Filtrar los períodos de interés
periodos_interes = [201901, 201902, 201903, 201904, 201905, 201906,
                    201907, 201908, 201909, 201910, 201911, 201912]
sell_agrup_filtrado_pi = sell_agrup_filtrado[sell_agrup_filtrado['periodo'].isin(periodos_interes)]

In [21]:
# Hacer predicciones para todos los product_id en sell_agrup_filtrado_pi usando modelos por cluster
print("=== APLICANDO PREDICCIONES CON MODELOS POR CLUSTER ===")

# Verificar que tenemos los modelos por cluster disponibles
if 'modelos_por_cluster' in locals() and len(modelos_por_cluster) > 0:
    print(f"Modelos disponibles para clusters: {list(modelos_por_cluster.keys())}")
    
    # Crear dataset para predicciones
    predicciones = []
    
    # Obtener todos los product_id únicos
    productos_unicos = sell_agrup_filtrado_pi['product_id'].unique()
    print(f"Total productos a predecir: {len(productos_unicos)}")
    
    # Estadísticas de métodos usados
    contador_metodos = {
        'Modelo Cluster': 0,
        'Promedio Periodos': 0
    }
    
    for product_id in productos_unicos:
        # Filtrar datos del producto específico
        datos_producto = sell_agrup_filtrado_pi[sell_agrup_filtrado_pi['product_id'] == product_id].copy()
        datos_producto = datos_producto.sort_values('periodo')
        
        # Verificar si tiene datos para todos los 12 períodos (201901 a 201912)
        periodos_completos = [201901, 201902, 201903, 201904, 201905, 201906,
                             201907, 201908, 201909, 201910, 201911, 201912]
        
        periodos_disponibles = set(datos_producto['periodo'].tolist())
        tiene_todos_periodos = all(p in periodos_disponibles for p in periodos_completos)
        
        # Buscar el cluster del producto en el dataset con clusters
        cluster_producto = None
        if product_id in clusters['product_id'].values:
            cluster_producto = clusters[clusters['product_id'] == product_id]['cluster'].iloc[0]
        
        # Variables para tracking
        metodo_usado = None
        prediccion = None
        cluster_usado = None
        
        # LÓGICA DE PREDICCIÓN
        if (tiene_todos_periodos and 
            cluster_producto is not None and 
            pd.notna(cluster_producto) and 
            cluster_producto in modelos_por_cluster):
            
            # CASO 1: Usar modelo específico del cluster
            try:
                # Obtener valores específicos para la predicción
                tn_201912 = datos_producto[datos_producto['periodo'] == 201912]['tn'].iloc[0]
                tn_201911 = datos_producto[datos_producto['periodo'] == 201911]['tn'].iloc[0]
                tn_201910 = datos_producto[datos_producto['periodo'] == 201910]['tn'].iloc[0]
                tn_201909 = datos_producto[datos_producto['periodo'] == 201909]['tn'].iloc[0]
                tn_201908 = datos_producto[datos_producto['periodo'] == 201908]['tn'].iloc[0]
                tn_201907 = datos_producto[datos_producto['periodo'] == 201907]['tn'].iloc[0]
                tn_201906 = datos_producto[datos_producto['periodo'] == 201906]['tn'].iloc[0]
                tn_201905 = datos_producto[datos_producto['periodo'] == 201905]['tn'].iloc[0]
                tn_201904 = datos_producto[datos_producto['periodo'] == 201904]['tn'].iloc[0]
                tn_201903 = datos_producto[datos_producto['periodo'] == 201903]['tn'].iloc[0]
                tn_201902 = datos_producto[datos_producto['periodo'] == 201902]['tn'].iloc[0]
                tn_201901 = datos_producto[datos_producto['periodo'] == 201901]['tn'].iloc[0]
                
                # Verificar que no haya valores NaN en las variables predictoras
                valores_predictores = [tn_201912, tn_201911, tn_201910, tn_201909, tn_201908, 
                                     tn_201907, tn_201906, tn_201905, tn_201904, tn_201903, 
                                     tn_201902, tn_201901]
                
                if all(pd.notna(val) for val in valores_predictores):
                    # Crear array de características para el modelo
                    X_pred = np.array(valores_predictores).reshape(1, -1)
                    
                    # Usar el modelo específico del cluster
                    modelo_cluster = modelos_por_cluster[cluster_producto]
                    prediccion = modelo_cluster.predict(X_pred)[0]
                    metodo_usado = "Modelo Cluster"
                    cluster_usado = cluster_producto
                    contador_metodos['Modelo Cluster'] += 1
                else:
                    # Hay NaN en predictores, usar promedio
                    prediccion = datos_producto['tn'].mean()
                    metodo_usado = "Promedio Periodos"
                    contador_metodos['Promedio Periodos'] += 1
                    
            except Exception as e:
                print(f"⚠️  Error aplicando modelo cluster para producto {product_id}: {e}")
                # Fallback a promedio
                prediccion = datos_producto['tn'].mean()
                metodo_usado = "Promedio Periodos"
                contador_metodos['Promedio Periodos'] += 1
        else:
            # CASO 2: Usar promedio de los períodos disponibles
            # (producto sin cluster, cluster sin modelo, o datos incompletos)
            prediccion = datos_producto['tn'].mean()
            metodo_usado = "Promedio Periodos"
            contador_metodos['Promedio Periodos'] += 1
            
            # Razón específica
            if not tiene_todos_periodos:
                razon = "Datos incompletos"
            elif cluster_producto is None or pd.isna(cluster_producto):
                razon = "Sin cluster asignado"
            elif cluster_producto not in modelos_por_cluster:
                razon = "Cluster sin modelo"
            else:
                razon = "Otra razón"
        
        # Agregar resultado
        predicciones.append({
            'product_id': product_id,
            'prediccion_202002': prediccion,
            'metodo_usado': metodo_usado,
            'cluster_usado': cluster_usado,
            'periodos_disponibles': len(datos_producto),
            'tiene_12_periodos': tiene_todos_periodos
        })
    
    # Crear DataFrame con las predicciones
    df_predicciones = pd.DataFrame(predicciones)
    
    print(f"\n=== RESUMEN DE PREDICCIONES ===")
    print(f"Total productos procesados: {len(df_predicciones)}")
    print(f"Productos con modelo cluster: {contador_metodos['Modelo Cluster']}")
    print(f"Productos con promedio: {contador_metodos['Promedio Periodos']}")
    
    # Estadísticas por cluster usado
    if contador_metodos['Modelo Cluster'] > 0:
        print(f"\n=== PREDICCIONES POR CLUSTER ===")
        clusters_usados = df_predicciones[df_predicciones['metodo_usado'] == 'Modelo Cluster']['cluster_usado'].value_counts()
        for cluster_id, count in clusters_usados.items():
            print(f"Cluster {cluster_id}: {count} productos")
    
    print(f"\n=== PRIMERAS PREDICCIONES ===")
    print(df_predicciones[['product_id', 'prediccion_202002', 'metodo_usado', 'cluster_usado']].head(10))
    
    # Mostrar estadísticas de las predicciones
    print(f"\n=== ESTADÍSTICAS DE PREDICCIONES ===")
    print(f"Predicción mínima: {df_predicciones['prediccion_202002'].min():.2f}")
    print(f"Predicción máxima: {df_predicciones['prediccion_202002'].max():.2f}")
    print(f"Predicción promedio: {df_predicciones['prediccion_202002'].mean():.2f}")
    print(f"Desviación estándar: {df_predicciones['prediccion_202002'].std():.2f}")
    
    # Comparar predicciones por método
    pred_cluster = df_predicciones[df_predicciones['metodo_usado'] == 'Modelo Cluster']['prediccion_202002']
    pred_promedio = df_predicciones[df_predicciones['metodo_usado'] == 'Promedio Periodos']['prediccion_202002']
    
    if len(pred_cluster) > 0:
        print(f"\n=== COMPARACIÓN POR MÉTODO ===")
        print(f"Predicciones Modelo Cluster - Media: {pred_cluster.mean():.2f}, Std: {pred_cluster.std():.2f}")
    if len(pred_promedio) > 0:
        print(f"Predicciones Promedio - Media: {pred_promedio.mean():.2f}, Std: {pred_promedio.std():.2f}")
    
else:
    print("❌ Error: Los modelos por cluster no están disponibles.")
    print("   Ejecuta primero la celda de entrenamiento de modelos por cluster.")

=== APLICANDO PREDICCIONES CON MODELOS POR CLUSTER ===
Modelos disponibles para clusters: [np.int64(0), np.int64(3), np.int64(7)]
Total productos a predecir: 780


c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\s


=== RESUMEN DE PREDICCIONES ===
Total productos procesados: 780
Productos con modelo cluster: 537
Productos con promedio: 243

=== PREDICCIONES POR CLUSTER ===
Cluster 0.0: 452 productos
Cluster 7.0: 72 productos
Cluster 3.0: 13 productos

=== PRIMERAS PREDICCIONES ===
   product_id  prediccion_202002       metodo_usado  cluster_usado
0       20001        1486.127784     Modelo Cluster            7.0
1       20002         222.751485     Modelo Cluster            3.0
2       20003         329.876105     Modelo Cluster            3.0
3       20004         586.072646     Modelo Cluster            7.0
4       20005         604.941473     Modelo Cluster            0.0
5       20006         484.367225     Modelo Cluster            0.0
6       20007         405.638033     Modelo Cluster            0.0
7       20008         367.285518     Modelo Cluster            0.0
8       20009         541.322587  Promedio Periodos            NaN
9       20010         389.631100     Modelo Cluster        

c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\cacic\anaconda3\envs\ldi2\Lib\s

In [22]:
# Análisis detallado de las predicciones por cluster
print("=== ANÁLISIS DETALLADO POR CLUSTER ===")

if 'df_predicciones' in locals() and len(df_predicciones) > 0:
    
    # Crear análisis por cluster
    print(f"\n📊 DISTRIBUCIÓN DE PRODUCTOS POR MÉTODO:")
    metodo_counts = df_predicciones['metodo_usado'].value_counts()
    for metodo, count in metodo_counts.items():
        porcentaje = (count / len(df_predicciones)) * 100
        print(f"  {metodo}: {count} productos ({porcentaje:.1f}%)")
    
    # Análisis específico para productos con modelo cluster
    productos_cluster = df_predicciones[df_predicciones['metodo_usado'] == 'Modelo Cluster'].copy()
    
    if len(productos_cluster) > 0:
        print(f"\n🎯 ANÁLISIS DE PREDICCIONES POR CLUSTER:")
        print(f"{'Cluster':<8} {'Productos':<10} {'Pred. Media':<12} {'Pred. Min':<10} {'Pred. Max':<10} {'Std':<8}")
        print("-" * 65)
        
        for cluster_id in sorted(productos_cluster['cluster_usado'].unique()):
            productos_del_cluster = productos_cluster[productos_cluster['cluster_usado'] == cluster_id]
            predicciones_cluster = productos_del_cluster['prediccion_202002']
            
            print(f"{cluster_id:<8} {len(productos_del_cluster):<10} "
                  f"{predicciones_cluster.mean():<12.2f} "
                  f"{predicciones_cluster.min():<10.2f} "
                  f"{predicciones_cluster.max():<10.2f} "
                  f"{predicciones_cluster.std():<8.2f}")
        
        # Encontrar cluster con predicciones más altas y más bajas
        cluster_stats = productos_cluster.groupby('cluster_usado')['prediccion_202002'].agg(['mean', 'count']).reset_index()
        cluster_mayor = cluster_stats.loc[cluster_stats['mean'].idxmax()]
        cluster_menor = cluster_stats.loc[cluster_stats['mean'].idxmin()]
        
        print(f"\n🔝 CLUSTER CON PREDICCIONES MÁS ALTAS: {cluster_mayor['cluster_usado']} "
              f"(media: {cluster_mayor['mean']:.2f}, productos: {cluster_mayor['count']})")
        print(f"📉 CLUSTER CON PREDICCIONES MÁS BAJAS: {cluster_menor['cluster_usado']} "
              f"(media: {cluster_menor['mean']:.2f}, productos: {cluster_menor['count']})")
    
    # Análisis de productos con promedio
    productos_promedio = df_predicciones[df_predicciones['metodo_usado'] == 'Promedio Periodos'].copy()
    
    if len(productos_promedio) > 0:
        print(f"\n📈 ANÁLISIS DE PREDICCIONES POR PROMEDIO:")
        print(f"Total productos: {len(productos_promedio)}")
        print(f"Predicción promedio: {productos_promedio['prediccion_202002'].mean():.2f}")
        print(f"Desviación estándar: {productos_promedio['prediccion_202002'].std():.2f}")
        print(f"Rango: {productos_promedio['prediccion_202002'].min():.2f} - {productos_promedio['prediccion_202002'].max():.2f}")
        
        # Razones por las que se usó promedio
        productos_sin_12_periodos = productos_promedio[~productos_promedio['tiene_12_periodos']]
        print(f"\nRazones para usar promedio:")
        print(f"  - Datos incompletos (<12 períodos): {len(productos_sin_12_periodos)} productos")
        print(f"  - Sin cluster o cluster sin modelo: {len(productos_promedio) - len(productos_sin_12_periodos)} productos")
    
    # Comparación general
    if len(productos_cluster) > 0 and len(productos_promedio) > 0:
        print(f"\n⚖️  COMPARACIÓN MÉTODO CLUSTER vs PROMEDIO:")
        print(f"Predicción media - Cluster: {productos_cluster['prediccion_202002'].mean():.2f}")
        print(f"Predicción media - Promedio: {productos_promedio['prediccion_202002'].mean():.2f}")
        diferencia = productos_cluster['prediccion_202002'].mean() - productos_promedio['prediccion_202002'].mean()
        print(f"Diferencia: {diferencia:.2f} ({diferencia/productos_promedio['prediccion_202002'].mean()*100:.1f}%)")
    
    # Productos con predicciones extremas
    print(f"\n🎯 PRODUCTOS CON PREDICCIONES EXTREMAS:")
    top_5_max = df_predicciones.nlargest(5, 'prediccion_202002')
    top_5_min = df_predicciones.nsmallest(5, 'prediccion_202002')
    
    print(f"\nTOP 5 PREDICCIONES MÁS ALTAS:")
    for _, row in top_5_max.iterrows():
        cluster_info = f"(Cluster {row['cluster_usado']})" if pd.notna(row['cluster_usado']) else "(Sin cluster)"
        print(f"  Producto {row['product_id']}: {row['prediccion_202002']:.2f} - {row['metodo_usado']} {cluster_info}")
    
    print(f"\nTOP 5 PREDICCIONES MÁS BAJAS:")
    for _, row in top_5_min.iterrows():
        cluster_info = f"(Cluster {row['cluster_usado']})" if pd.notna(row['cluster_usado']) else "(Sin cluster)"
        print(f"  Producto {row['product_id']}: {row['prediccion_202002']:.2f} - {row['metodo_usado']} {cluster_info}")

else:
    print("❌ No hay predicciones disponibles para analizar.")

=== ANÁLISIS DETALLADO POR CLUSTER ===

📊 DISTRIBUCIÓN DE PRODUCTOS POR MÉTODO:
  Modelo Cluster: 537 productos (68.8%)
  Promedio Periodos: 243 productos (31.2%)

🎯 ANÁLISIS DE PREDICCIONES POR CLUSTER:
Cluster  Productos  Pred. Media  Pred. Min  Pred. Max  Std     
-----------------------------------------------------------------
0.0      452        39.18        7.88       604.94     66.79   
3.0      13         102.18       -0.88      343.86     122.81  
7.0      72         79.45        -41.74     1486.13    193.10  

🔝 CLUSTER CON PREDICCIONES MÁS ALTAS: 3.0 (media: 102.18, productos: 13.0)
📉 CLUSTER CON PREDICCIONES MÁS BAJAS: 0.0 (media: 39.18, productos: 452.0)

📈 ANÁLISIS DE PREDICCIONES POR PROMEDIO:
Total productos: 243
Predicción promedio: 22.24
Desviación estándar: 61.75
Rango: 0.03 - 590.33

Razones para usar promedio:
  - Datos incompletos (<12 períodos): 130 productos
  - Sin cluster o cluster sin modelo: 113 productos

⚖️  COMPARACIÓN MÉTODO CLUSTER vs PROMEDIO:
Predicc

In [23]:
# Preparar dataset para exportar con las columnas requeridas
df_export = df_predicciones[['product_id', 'prediccion_202002']].copy()

# Renombrar la columna prediccion_202002 a tn
df_export = df_export.rename(columns={'prediccion_202002': 'tn'})

# Mostrar el dataset que se va a exportar
print("Dataset a exportar:")
print(df_export.head(10))
print(f"\nShape del dataset: {df_export.shape}")
print(f"Columnas: {list(df_export.columns)}")

# Definir la ruta de salida
drive_base_path = 'C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression'
filename = 'predicciones_202002_dtw_2018.csv'
filepath = os.path.join(drive_base_path, filename)

# Crear el directorio si no existe
os.makedirs(drive_base_path, exist_ok=True)

# Guardar el archivo CSV separado por comas
df_export.to_csv(filepath, sep=',', index=False)

print(f"\nArchivo guardado exitosamente en: {filepath}")
print(f"Total de registros guardados: {len(df_export)}")

# Verificar que el archivo se guardó correctamente
if os.path.exists(filepath):
    file_size = os.path.getsize(filepath)
    print(f"Tamaño del archivo: {file_size} bytes")
    
    # Leer las primeras líneas para verificar el formato
    with open(filepath, 'r') as f:
        primeras_lineas = [f.readline().strip() for _ in range(5)]
    
    print(f"\nPrimeras líneas del archivo CSV:")
    for i, linea in enumerate(primeras_lineas):
        print(f"Línea {i+1}: {linea}")
else:
    print("Error: El archivo no se pudo crear.")

Dataset a exportar:
   product_id           tn
0       20001  1486.127784
1       20002   222.751485
2       20003   329.876105
3       20004   586.072646
4       20005   604.941473
5       20006   484.367225
6       20007   405.638033
7       20008   367.285518
8       20009   541.322587
9       20010   389.631100

Shape del dataset: (780, 2)
Columnas: ['product_id', 'tn']

Archivo guardado exitosamente en: C:/Repositorios-Ing.Carlos-Cicconi/labo3-2025r/Outputs/Linear Regression\predicciones_202002_dtw_2018.csv
Total de registros guardados: 780
Tamaño del archivo: 19309 bytes

Primeras líneas del archivo CSV:
Línea 1: product_id,tn
Línea 2: 20001,1486.1277836403226
Línea 3: 20002,222.75148494105372
Línea 4: 20003,329.8761049831081
Línea 5: 20004,586.0726459439982
